# Aula 16 · Séries temporais

Esta aula apresenta o [capítulo 16 do site](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/). A ideia central: **uma série no tempo é a soma de uma tendência, de ciclos e de ruído** — separar as partes com médias e retas é o que permite prever o próximo valor, e medir o quanto se erra.

**Ao fim da aula você consegue:**

1. reconhecer tendência, ciclos e ruído no gráfico de uma série;
2. calcular a média móvel, a tendência e o perfil de um ciclo;
3. deduzir e aplicar a suavização exponencial;
4. comparar previsões com MAE e RMSE, em dados de teste.

**Roteiro:** 🧩 · 1. a série · 2. média móvel · 3. tendência · 4. sazonalidade · 5. 🧑‍🏫 suavização · 6. prever e medir · 7. outra área · 🎯 prática · 🧩 o pico · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

# --- dados desta aula (baixados do site, se ainda não estiverem aqui) ---
import os
import urllib.request

for ARQUIVO in ["consumo_energia.csv"]:
    if not os.path.exists(ARQUIVO):
        urllib.request.urlretrieve("https://lacouth.github.io/metodos_telecom-site/dados/" + ARQUIVO, ARQUIVO)
    print(ARQUIVO, "pronto")

## 🧩 O problema da aula

> **Setor elétrico — o pico da semana que vem.**
>
> *A distribuidora de uma cidade pequena contrata energia com uma semana de
> antecedência, e paga caro se o consumo passar do contratado. A engenheira de
> planejamento tem 8 semanas de consumo, de hora em hora, e pergunta: "**qual vai ser o
> pico da semana que vem, e em que dia e hora?**"*

No fim da aula, você prevê a semana inteira juntando a tendência e o ciclo semanal.

## 1. Uma série temporal

📖 [capítulo 16 · Uma série temporal](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#uma-serie-temporal)

In [ ]:
# 📦 dados prontos — só rode esta célula
dados = np.loadtxt("consumo_energia.csv", delimiter=",", skiprows=1)
hora = dados[:, 0]
consumo = dados[:, 1]      # MW, de hora em hora, 8 semanas (começa numa segunda à 0 h)

**✍️ Passo 1.** Desenhe o consumo das 8 semanas (`hora / 24` no eixo x, para ver em dias) e, numa segunda figura, só a primeira semana (`consumo[:168]`).

In [ ]:
# ✍️ passo 1

**Preveja:** que padrões aparecem se repetindo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Um **ciclo diário** (vale de madrugada, pico à noite), um **ciclo semanal** (sábado e
domingo mais baixos), uma leve **subida** ao longo das semanas e **ruído**.

📖 [capítulo 16 · Uma série temporal](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#uma-serie-temporal)

</details>

## 2. Média móvel

📖 [capítulo 16 · Média móvel](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#media-movel)

### 🎯 Sua vez — A média móvel

Escreva `media_movel(serie, janela)`, que devolve o array das médias de cada `janela` valores seguidos.

In [ ]:
def media_movel(serie, janela):
    # sua solução aqui
    pass

In [ ]:
confere(media_movel, [
    ((np.array([10.0, 12.0, 11.0, 15.0, 14.0, 16.0]), 3), [11.0, 38 / 3, 40 / 3, 15.0]),
])

<details>
<summary><b>💡 Dica</b></summary>

`np.zeros(len(serie) - janela + 1)` e, para cada `i`, `np.mean(serie[i:i + janela])`.

</details>

**✍️ Passo 2.** Desenhe o consumo junto com a sua média móvel de **24 horas** e, depois, com a de **168 horas** (uma semana).

In [ ]:
# ✍️ passo 2

**Preveja:** qual das duas mostra a tendência limpa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A de 168 h: ela apaga os dois ciclos e sobe devagar, de 52 para 57 MW. A de 24 h
ainda "balança" com os fins de semana. A janela tem de ter o tamanho de um ciclo
inteiro.

📖 [capítulo 16 · Média móvel](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#media-movel)

</details>

## 3. Tendência

📖 [capítulo 16 · Tendência](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#tendencia)

**✍️ Passo 3.** Calcule a média de cada dia (um laço em `d`, com `np.mean(consumo[24*d:24*d + 24])`) e ajuste uma reta a elas com `np.polyfit`.

In [ ]:
# ✍️ passo 3

**Preveja:** quantos MW por dia o consumo cresce?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cerca de 0,08 MW por dia — uns 29 MW por ano, se continuar: mais da metade do
consumo de hoje. É o número que faz uma distribuidora planejar uma subestação.

📖 [capítulo 16 · Tendência](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#tendencia)

</details>

## 4. Sazonalidade

O "dia típico": a média de cada hora do dia em todos os dias. A fatia `consumo[h::24]`
pega a hora `h` de todos os dias.

📖 [capítulo 16 · Sazonalidade](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#sazonalidade)

**✍️ Passo 4.** Monte o perfil de 24 horas (`np.mean(consumo[h::24])` para cada `h`) e desenhe-o.

In [ ]:
# ✍️ passo 4

**Preveja:** em que hora é o pico? E o vale?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Pico às **19 h** (63 MW), vale às **4 h** (46 MW). O pico é o do chuveiro e da
iluminação, quando as pessoas chegam em casa.

📖 [capítulo 16 · Sazonalidade](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#sazonalidade)

</details>

## 5. No quadro: a suavização exponencial

📖 [capítulo 16 · No quadro: a suavização exponencial](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#no-quadro-a-suavizacao-exponencial)

### 🧑‍🏫 No quadro — a suavização exponencial simples

Caderno de papel aberto. No quadro:

1. a previsão nova como mistura do dado novo com a previsão anterior;
2. substituir a fórmula nela mesma, e de novo;
3. os pesos $\alpha$, $\alpha(1-\alpha)$, $\alpha(1-\alpha)^2$...;
4. o papel de $\alpha$: reagir rápido ou ser estável.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ \hat y_{t+1} = \alpha\,y_t + (1 - \alpha)\,\hat y_t
 = \alpha\,y_t + \alpha(1-\alpha)\,y_{t-1} + \alpha(1-\alpha)^2\,y_{t-2} + \cdots $$

</details>

**✍️ Passo 5.** Aplique a suavização às médias diárias do passo 3 com `alpha = 0.5`: comece com `previsao = media_dia[0]` e atualize para cada dia. Imprima a previsão para o dia seguinte ao último.

In [ ]:
# ✍️ passo 5

**Preveja:** a previsão vai ficar perto da média do último dia (um domingo) ou da média da semana?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

No meio: 53,9 MW, puxada para baixo pelo fim de semana. A suavização simples não
sabe que segunda-feira volta a subir — ela não conhece ciclos.

📖 [capítulo 16 · No quadro: a suavização exponencial](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#no-quadro-a-suavizacao-exponencial)

</details>

## 6. Prever e medir o erro

Treino: as 7 primeiras semanas. Teste: a última, que o método não pode ver.

📖 [capítulo 16 · Prever e medir o erro](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#prever-e-medir-o-erro)

### 🎯 Sua vez — O erro absoluto médio

Escreva `mae(previsto, real)`, a média de `abs(previsto - real)` (os dois são arrays).

In [ ]:
def mae(previsto, real):
    # sua solução aqui
    pass

In [ ]:
confere(mae, [
    ((np.array([10.0, 12.0, 11.0]), np.array([11.0, 12.0, 14.0])), 4 / 3),
])

<details>
<summary><b>💡 Dica</b></summary>

Uma linha, com `np.mean` e `abs`.

</details>

**✍️ Passo 6.** Com `treino = consumo[:7 * 168]` e `teste = consumo[7 * 168:]`, calcule o MAE da previsão ingênua (`np.zeros(168) + treino[-1]`) e da sazonal (`treino[-168:]`, a semana passada).

In [ ]:
# ✍️ passo 6

**Preveja:** copiar a semana passada é melhor que repetir o último valor?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Muito: 1,8 MW contra 6,4 MW. Um método simples que **respeita os ciclos** bate um
que os ignora. Somando a tendência de uma semana, o erro cai mais um pouco.

📖 [capítulo 16 · Prever e medir o erro](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#prever-e-medir-o-erro)

</details>

> ⚠️ **Armadilha.** Avaliar um método nos mesmos dados que ele usou para prever dá um erro bom demais.
O teste precisa ser o **futuro** que o método não viu.

## 7. Mesmo método, outra área

**Clima.** A temperatura mensal de João Pessoa tem um ciclo de 12 meses: o método
sazonal usa o mesmo mês do ano passado.

📖 [capítulo 16 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#mesmo-metodo-outra-area)

In [ ]:
# 📦 dados prontos — só rode esta célula
temperatura = np.array([27.6, 27.8, 27.6, 27.3, 26.6, 25.7, 25.1, 25.2, 25.9, 26.6, 27.0, 27.3,
                        27.8, 28.0, 27.9, 27.4, 26.8, 25.8, 25.3, 25.3, 26.0, 26.8, 27.2, 27.6,
                        28.0, 28.1, 28.0, 27.7, 26.9, 26.1, 25.4, 25.6, 26.2, 26.9, 27.4, 27.7])

**✍️ Passo 7.** Preveja o terceiro ano (`temperatura[24:]`) com o segundo (`temperatura[12:24]`) e calcule o MAE com a sua função.

In [ ]:
# ✍️ passo 7

**Preveja:** o erro vai ser de décimos ou de graus?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Décimos: 0,17 °C. O mesmo método do consumo de energia, com outro período.

📖 [capítulo 16 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é o problema, logo abaixo: juntar a tendência e o ciclo numa previsão.

## 🧩 Resolvendo o problema

> *"**Qual vai ser o pico da semana que vem, e em que dia e hora?**"* — a engenheira.

A previsão combina as duas partes: a **semana passada** (o ciclo) mais o **crescimento
de uma semana** (a tendência, 168 horas vezes a inclinação da reta ajustada a toda a
série).

### 🎯 Sua vez — O pico da próxima semana

Escreva `pico_da_proxima_semana(serie)`, que ajusta uma reta à série inteira (com
o tempo `np.linspace(0, n - 1, n)`), monta a previsão (as últimas 168 horas mais
`inclinação * 168`) e devolve a tupla `(maior valor previsto, posição dele na
semana)`.

In [ ]:
def pico_da_proxima_semana(serie):
    # sua solução aqui
    pass

In [ ]:
confere(pico_da_proxima_semana, [
    ((consumo,), (71.2914475900096, 43)),
])

<details>
<summary><b>💡 Dica</b></summary>

`coef = np.polyfit(...)`; a previsão é `serie[n - 168:] + coef[0] * 168`; depois `np.max` e `np.argmax`.

</details>

In [ ]:
resposta = pico_da_proxima_semana(consumo)
if resposta is not None:
    pico, k = resposta
    dias = ["segunda", "terça", "quarta", "quinta", "sexta", "sábado", "domingo"]
    print("pico previsto:", pico, "MW, na", dias[k // 24], "às", k % 24, "h")

<details>
<summary><b>▶ O que os números dizem</b></summary>

O pico previsto é de **71.3 MW**, na terça às
19 h. A engenheira contrataria um pouco acima disso, com uma
**margem**: o erro típico do método, medido no teste (RMSE de cerca de 2,2 MW), diz
quanto.

Um detalhe honesto: o pico previsto é o pico de **uma** semana passada, com o ruído
dela. Uma previsão mais estável usaria o perfil médio de várias semanas (o ciclo sem o
ruído) mais a tendência — e a margem cuidaria do resto.

</details>

## 📋 A lista

Abra a [Lista 16](https://lacouth.github.io/metodos_telecom-site/listas/lista16/). O **Exercício 01** é à mão (✏️): média móvel e suavização
exponencial numa série curta. Comece por ele, no papel.

**a)** Qual a primeira média móvel de janela 3 da série 10, 12, 11, 15, 14, 16?

<details>
<summary><b>▶ Resposta</b></summary>

$(10 + 12 + 11)/3 = 11$.

</details>

Termine o exercício e siga para o **Exercício 02**, a média móvel como função.

## 🚪 Antes de sair

**1.** Por que a janela da média móvel precisa ter o tamanho de um ciclo inteiro?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Para que as partes altas e baixas do ciclo se compensem dentro da janela; com meio ciclo, a média ainda oscila.

</details>

**2.** O que $\alpha$ controla na suavização exponencial?

<details>
<summary><b>▶ Resposta da 2</b></summary>

O peso do dado mais recente: perto de 1, a previsão segue o último valor (rápida, mas tremida); perto de 0, lembra de muito tempo atrás (estável, mas lenta).

</details>

**3.** Por que copiar a semana passada bateu o método ingênuo por tanto?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Porque a série tem um ciclo semanal forte: a mesma hora da semana passada já traz o pico da noite e o vale da madrugada, que o valor repetido ignora.

</details>

## 🏠 Para casa

- Termine a [Lista 16](https://lacouth.github.io/metodos_telecom-site/listas/lista16/).
- Leia o começo do [capítulo 17](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/17-fft/): e se não
  soubermos de antemão qual é o período do ciclo?